# Leçon 18 : Sécuriser les agents IA avec des reçus cryptographiques

## Cahier pratique

Ce cahier présente quatre exercices :

1. **Signer votre premier reçu** pour un appel d'outil agent et le vérifier.
2. **Altérer le reçu** et observer l'échec de la vérification.
3. **Construire une chaîne de trois reçus** et confirmer l'intégrité de la chaîne.
4. **Envelopper un appel d’outil Microsoft Agent Framework** pour que chaque action émette un reçu.

Toutes les primitives cryptographiques sont importées de bibliothèques bien maintenues (`pynacl` pour Ed25519, `jcs` pour JSON canonique RFC 8785, `hashlib` de la bibliothèque standard Python pour le SHA-256). La logique du reçu elle-même est du Python simple que vous pouvez lire et modifier.

Exécutez les cellules dans l’ordre. Chaque section est courte et autonome.


## Installation

Installez les deux dépendances. Les deux ont des licences permissives (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Utilitaires d'aide

Ces deux utilitaires gèrent le codage base64url (sans remplissage) et le hachage SHA-256 d'objets arbitraires. Ils permettent de garder le reste du carnet concentré sur la logique du reçu elle-même.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1 : Signez votre premier reçu

Imaginez que notre agent pour **Contoso Travel** vient de rechercher des vols de Sydney à Los Angeles pour un client. Nous voulons enregistrer l'affirmation de la passerelle concernant cet appel d'outil afin qu'un auditeur futur puisse la vérifier sous une clé d'émetteur épinglée séparément.

### Étape 1.1 : Générer une clé de signature

En production, la clé de signature de l'agent résiderait dans un module de sécurité matériel (HSM), Azure Key Vault ou un magasin protégé similaire. Pour cette leçon, nous générons une clé fraîche en mémoire.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Étape 1.2 : Construire la charge utile du reçu

La charge utile contient tout ce que nous voulons que le reçu atteste : qui a agi, quel outil, avec quels arguments, ce qui a été renvoyé, sous quelle politique, et quand. Nous hachons les arguments et le résultat plutôt que de les inclure en ligne afin que le reçu ne divulgue pas de contenu sensible.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Étape 1.3 : Signer et assembler le reçu

Trois étapes :

1. Canonicaliser la charge utile à l’aide de JCS afin que deux implémentations produisant le même reçu logique produisent des octets identiques au niveau des bits.
2. Signer directement les octets canoniques avec la clé privée Ed25519. PureEdDSA effectue un hachage du message en interne, donc un pré-hachage supplémentaire modifierait le protocole.

La signature est ensuite jointe à la charge utile originale pour produire le reçu final.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Étape 1.4 : Vérifier le reçu

La vérification inverse le processus. Nous retirons la signature, recalculons les octets canoniques et vérifions la signature avec la clé publique de l’émetteur.

La vérification est hors ligne : aucun service à appeler et aucun accès réseau. Elle nécessite tout de même une chose en plus du reçu : la clé publique de l’émetteur, obtenue via un canal que l’auditeur a déjà confiance, comme un ensemble JWK publié par l’émetteur ou une valeur préalablement ancrée. Le champ `public_key` à l’intérieur du reçu ne peut jouer ce rôle, car celui qui écrit le reçu écrit aussi ce champ. La section 2 montre la falsification que cela empêche.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Vous devriez voir `Receipt is valid: True`, `Pre-hashed receipt valid: False` et `Malformed receipts rejected: True`. Le cas positif prouve que le chemin de signature direct-JCS fonctionne ; le contrôle négatif rend la règle de portée de signature exécutable au lieu de la laisser sous forme de prose ; la dernière ligne montre une entrée mal formée renvoyant `False` au lieu de lever une exception.


## Section 2 : Falsifier le reçu

Le but même des reçus est qu’ils soient infalsifiables. Prouvons-le.

Nous allons modifier exactement un caractère du reçu et observer l’échec de la vérification.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Que vient-il de se passer ?

Lorsque nous avons modifié `policy_id`, les octets canoniques ont changé. La signature (qui portait sur les octets canoniques originaux) ne correspond plus. La vérification renvoie correctement `False`.

Il est impossible de modifier un champ signé, c’est-à-dire tout ce qui est en dehors de l’objet `signature`, et que le reçu soit toujours validé par la clé publique de l’émetteur, à moins que l’attaquant possède la clé privée de l’émetteur. L’objet `signature` lui-même n’est pas signé : `sig` est la signature, et `verify_receipt` ignore `public_key`, donc modifier ce champ seul ne change rien. Tant que la clé privée est stockée dans un coffre-fort de clés et que les vérificateurs utilisent la clé publiée indépendamment par l’émetteur, il est impossible de cacher une modification d’un champ signé. Cette condition est importante : la cellule suivante montre un reçu modifié qui est parfaitement validé avec une clé différente.

Essayez vous-même : modifiez le `tool_name` ou le `agent_id` ou le `timestamp` dans la cellule ci-dessus et réexécutez. Chaque modification d’un champ signé produit un reçu invalide.


### La falsification que la signature ne détecte pas

L'attaquant ci-dessus ne s'est pas ré-signé. Un plus prudent le ferait : altérerait le reçu,
le signerait avec une clé qu'il a générée lui-même, et mettrait cette clé dans le
champ `public_key`. Le résultat est un reçu dont la signature est entièrement authentique
pour son contenu.

Exécutez ceci et regardez laquelle des deux vérifications le détecte.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Vous devriez voir `True` puis `False`. Les trois lignes `assert`
font que c'est une vérification : la cellule génère une erreur si la falsification vérifie un jour contre la
clé de l'émetteur.

La falsification est interne cohérente. Sa signature correspond vraiment à son
contenu et à la clé qu'elle porte, donc un validateur qui lit la clé sur le
reçu la déclare valide. Rien ne semble incorrect dans le reçu, parce que
rien n'est incohérent.

La seule chose qui la détecte est la vérification contre une clé que l'attaquant n'a
pas pu influencer. C'est pourquoi `verify_receipt` prend la clé en argument : une
fonction avec la signature `verify_receipt(receipt)` ne peut pas être écrite en toute sécurité,
parce que la seule clé qu'elle pourrait atteindre est celle fournie par un attaquant.

Le même reçu est sauvegardé sous `sample_receipts/04_resigned_receipt.json` si vous
souhaitez l'inspecter hors du notebook.


## Section 3 : Chaîner les reçus ensemble

Un seul reçu protège une action. La plupart des agents effectuent de nombreuses actions. Pour rendre toute la séquence évidente en cas de falsification, nous relions chaque reçu au précédent en incluant le hachage du reçu précédent dans la charge utile du nouveau reçu.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Si quelqu'un supprime ou réorganise un reçu dans la séquence présentée, le lien suivant se casse. La détection de la troncature ou de la réécriture de suffixe contrôlée par le signataire nécessite également une tête courante fixée, un point de contrôle externe ou un engagement authentifié de la population.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Maintenant, brisez la chaîne en falsifiant le reçu du milieu et revérifiez. Le reçu falsifié échoue sa vérification de signature, ET le reçu suivant échoue sa vérification du lien en chaîne (parce que son `previous_receipt_hash` ne correspond plus au hash modifié du reçu du milieu).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Le reçu 0 est toujours vérifié (il n’a pas été modifié et n’a pas de prédécesseur dont il dépend). Le reçu 1 échoue à sa vérification de signature car nous avons changé `tool_args_hash`. Le reçu 2 échoue à sa vérification de chaîne car son `previous_receipt_hash` a été calculé à partir du reçu 1 original (maintenant modifié). 

Même si un attaquant ré-signe le reçu 1 modifié, la nouvelle signature échoue avec la clé de l’émetteur à moins qu’il ne détienne la clé privée de l’émetteur (une nouvelle signature avec leur propre clé échoue, comme l’a montré la section 2), et la non-correspondance de la chaîne dans le reçu 2 exposerait toujours la falsification. Pour cacher la modification, l’attaquant devrait ré-signer chaque reçu à partir du point de modification, ce qui nécessite la possession de la clé privée de l’émetteur.


## Section 4 : Envelopper un appel d'outil d'agent avec signature de reçu

Dans un déploiement réel, vous ne voulez pas que chaque auteur d'agent se souvienne d'appeler `make_receipt`. Vous voulez que la signature des reçus soit automatique pour chaque invocation d'outil.

Voici le modèle le plus simple : une classe wrapper qui prend une fonction d'outil callable et retourne une version émettant un reçu. Cela s'adapte à n'importe quel cadre d'agent, y compris le Microsoft Agent Framework (`agent_framework.foundry`).

Si vous n'avez pas de projet Microsoft Foundry configuré, la simulation locale ci-dessous démontre toujours le modèle.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Intégration avec le Microsoft Agent Framework

Le wrapper `ReceiptedTool` ci-dessus est indépendant du framework. Pour l’utiliser dans un agent construit avec le Microsoft Agent Framework, enregistrez la fonction encapsulée en tant qu'outil. Un croquis (vous remplaceriez le mock par un enregistrement réel d’un outil Microsoft Foundry) :

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Le framework d’agent n’a pas besoin de connaître quoi que ce soit sur les reçus. La signature des reçus est enveloppée autour de l’outil, et non intégrée au framework. C’est ainsi que vous ajoutez la provenance au code agent existant sans réécrire l’agent.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
